# Step 8: Improvement experiments (Jian + Claude, Tue 17 Nov night)
**Needs:** `experiments_v4.py`, `transfer_v3.py`, `features_v3.py`, `data_prep.py`, and the `rehearsal/` folder from notebook 05 (it reuses its 80% training split and its hidden 20%).

**What it tests** (each idea against the current frozen method, with the SAME support samples and seeds, so every comparison is paired):

| Idea | What it is |
|---|---|
| **D align** | Line up each Amsterdam feature with Madrid's distribution (no labels) before the Random Forest parts see it |
| **I year** | A Madrid model that predicts the construction YEAR feeds the stack layer (years mean the same in every city; class boundaries do not) |
| **H weights** | Trust Madrid pixels with high building coverage more, and years right on a class boundary less (training weights only) |
| **E boost** | A boosting model's probabilities added to the stack layer |
| **A rf** | Random Forest settings that memorise neighbourhoods less, chosen by Madrid spatial CV |
| **C prior** | Label-free correction of the class mix: estimate it from the predicted classes of the unlabelled pixels, re-weight once (the full EM version collapsed in testing because our blended probabilities are not calibrated) |
| **B smooth** | Other smoothing strengths / a wider window |

**Rules fixed BEFORE running (no changes after seeing results):**
1. Choosing happens ONLY on the 80% training split (Madrid: random + spatial CV; Amsterdam: n labels, rest scored). The hidden 20% neighbourhoods are used ONCE, at the end, to confirm.
2. Main score = the final method's score (with smoothing 0.5 / 3x3), averaged over 10 seeds.
3. An idea **passes** if: n=25 improves by at least +0.005 OR the average of the 5 n improves by at least +0.005, AND no single n gets worse by more than 0.005. Ideas that change Stage 1 (A, H) must also keep Madrid random CV within -0.005 (A additionally needs Madrid spatial CV +0.01).
4. Passed ideas are combined, the combination is checked, then confirmed on the hidden neighbourhoods. Only then would it replace the frozen method (by rerunning 00 FINAL with the new settings).

Prizes are all on AMSTERDAM F1 (average, peak n=200, low-data n=25), so the Amsterdam test decides. Madrid is checked so nothing breaks.

Writes only to `results/exp08/`. Run time: about 60 to 90 minutes.

In [1]:
import time, importlib, json
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score
import data_prep as dp, features_v3 as fv3, transfer_v3 as tv3, experiments_v4 as ex
for m in (dp, fv3, tv3, ex):
    importlib.reload(m)

SHOTS, TRIALS = (5, 25, 50, 100, 200), 10
SEED, CLASSES = tv3.SEED, tv3.CLASSES
R = Path("rehearsal")
for DATA_DIR in [Path("data"), Path("../data"), Path("../../data")]:
    if (DATA_DIR / "madrid_train.parquet").exists():
        break
OUT = Path("results/exp08"); OUT.mkdir(parents=True, exist_ok=True)
MAIN = ex.BASE_SMOOTH                      # (0.5, 1): the frozen final method
SPATIAL_BLOCK = 50                         # 1.5 km blocks, same as our spatial CV and notebook 05

t0 = time.time()
D = ex.load(R, DATA_DIR)
assert D["cols"] == json.load(open("feature_cols_v3.json")), "feature list differs"
print(f"80% split: Madrid {len(D['ym']):,} pixels, Amsterdam {len(D['ya']):,} pixels ({time.time()-t0:.0f}s)")

80% split: Madrid 61,731 pixels, Amsterdam 22,515 pixels (0s)


## 1. Madrid Stage 1 screening (ideas A and H, plus smoothing on Madrid)
Random CV = pixels mixed (like our official number). Spatial CV = whole 1.5 km blocks held out (like unseen neighbourhoods).

In [2]:
t0 = time.time()
BASE_RF = dict(tv3.RF_PARAMS)
madrid_cfgs = {
    "baseline":            (BASE_RF, False),
    "A_leaf5":             (dict(BASE_RF, min_samples_leaf=5), False),
    "A_leaf10":            (dict(BASE_RF, min_samples_leaf=10), False),
    "A_leaf20":            (dict(BASE_RF, min_samples_leaf=20), False),
    "A_leaf10_half":       (dict(BASE_RF, min_samples_leaf=10, max_samples=0.5), False),
    "H_weights":           (BASE_RF, True),
}
mrows = []
for name, (params, w) in madrid_cfgs.items():
    t = time.time()
    r = ex.madrid_cv(D, params, weights=w, block=SPATIAL_BLOCK)
    row = {"config": name, "random_cv": r["random"][0], "random_std": r["random"][1],
           "spatial_cv": r["spatial"][0], "spatial_std": r["spatial"][1]}
    for a, rad in ex.SMOOTH_VARIANTS[1:]:      # smoothing on the spatial out-of-fold probabilities
        Ps = tv3.smooth_proba(r["P_spatial"], D["ids_m"], alpha=a, r=rad)
        row[f"spatial_smooth_{a}_{rad}"] = f1_score(D["ym"], CLASSES[Ps.argmax(1)], average="macro")
    mrows.append(row)
    print(f"{name:15s} random {row['random_cv']:.4f} | spatial {row['spatial_cv']:.4f} | "
          f"spatial+smooth(0.5,3x3) {row['spatial_smooth_0.5_1']:.4f}  ({time.time()-t:.0f}s)")
mad = pd.DataFrame(mrows)
mad.round(4).to_csv(OUT / "madrid_screening.csv", index=False)

b = mad.set_index("config").loc["baseline"]
a_rows = mad[mad.config.str.startswith("A_")].set_index("config")
A_best = a_rows["spatial_cv"].idxmax()
A_madrid_ok = (a_rows.loc[A_best, "spatial_cv"] >= b.spatial_cv + 0.01) and (a_rows.loc[A_best, "random_cv"] >= b.random_cv - 0.005)
h = mad.set_index("config").loc["H_weights"]
H_madrid_ok = h.random_cv >= b.random_cv - 0.005
print(f"\nA: best = {A_best} (spatial {a_rows.loc[A_best,'spatial_cv']:.4f} vs {b.spatial_cv:.4f}) -> Madrid gate {'PASS' if A_madrid_ok else 'fail'}")
print(f"H: Madrid gate {'PASS' if H_madrid_ok else 'fail'}")
print(f"done in {time.time()-t0:.0f}s")

baseline        random 0.6623 | spatial 0.6165 | spatial+smooth(0.5,3x3) 0.6323  (124s)
A_leaf5         random 0.6546 | spatial 0.6137 | spatial+smooth(0.5,3x3) 0.6303  (114s)
A_leaf10        random 0.6454 | spatial 0.6102 | spatial+smooth(0.5,3x3) 0.6263  (106s)
A_leaf20        random 0.6353 | spatial 0.6050 | spatial+smooth(0.5,3x3) 0.6211  (97s)
A_leaf10_half   random 0.6379 | spatial 0.6071 | spatial+smooth(0.5,3x3) 0.6225  (63s)
H_weights       random 0.6588 | spatial 0.6157 | spatial+smooth(0.5,3x3) 0.6305  (114s)

A: best = A_leaf5 (spatial 0.6137 vs 0.6165) -> Madrid gate fail
H: Madrid gate PASS
done in 618s


## 2. Amsterdam transfer screening (the prize metric)
Every idea runs the full blend3 transfer for n = 5, 25, 50, 100, 200 x 10 seeds on the 80% split. C (class-mix correction) and B (smoothing variants) are post-processing, so they are scored on every run for free.

**Sanity check:** the baseline without smoothing must reproduce notebook 05's "inside training area" scores exactly.

In [3]:
t0 = time.time()
stage_cfgs = {"baseline": {}, "D_align": {"align": True}, "I_year": {"year": True},
              "H_weights": {"weights": True}, "E_boost": {"boost": True},
              "A_rf": {"rf_params": madrid_cfgs[A_best][0]}}

def screen(name, cfg):
    t = time.time()
    S = ex.build_stage1(D, cfg)
    rows = []
    for n in SHOTS:
        for k in range(TRIALS):
            sup, qry, P = ex.transfer_run(D, S, n, SEED + k)
            y, ids = D["ya"][qry], D["ids_a"][qry]
            for prior in (False, True):
                for (a, rad), f in ex.scores(P, y, ids, prior=prior).items():
                    rows.append({"config": name, "n_per_class": n, "seed": SEED + k, "prior": prior,
                                 "smooth": f"{a}_{rad}", "f1": f})
    df = pd.DataFrame(rows)
    main = df[(~df.prior) & (df.smooth == f"{MAIN[0]}_{MAIN[1]}")].groupby("n_per_class").f1.mean()
    print(f"{name:10s} " + " | ".join(f"n={n}: {main[n]:.4f}" for n in SHOTS) + f" | avg {main.mean():.4f}  ({time.time()-t:.0f}s)")
    return df

runs = pd.concat([screen(name, cfg) for name, cfg in stage_cfgs.items()], ignore_index=True)
runs.to_csv(OUT / "transfer_screening_all_runs.csv", index=False)

# sanity check against notebook 05 (same split, same seeds, same code path)
ref = pd.read_csv(R / "dress_rehearsal_all_runs.csv")
mine = runs[(runs.config == "baseline") & (~runs.prior) & (runs.smooth == "0.0_1")][["n_per_class", "seed", "f1"]]
chk = mine.merge(ref[["n_per_class", "seed", "f1_query_in_training_area"]], on=["n_per_class", "seed"])
gap = (chk.f1 - chk.f1_query_in_training_area).abs().max()
print(f"\nsanity check vs notebook 05: largest difference {gap:.6f} (rounding to 4 decimals in 05's file)")
assert gap < 1e-3, "baseline does not reproduce notebook 05: stop and check"
print(f"done in {time.time()-t0:.0f}s")

baseline   n=5: 0.5854 | n=25: 0.6619 | n=50: 0.6947 | n=100: 0.7118 | n=200: 0.7282 | avg 0.6764  (460s)
D_align    n=5: 0.6019 | n=25: 0.6612 | n=50: 0.6894 | n=100: 0.7072 | n=200: 0.7248 | avg 0.6769  (458s)
I_year     n=5: 0.5855 | n=25: 0.6618 | n=50: 0.6942 | n=100: 0.7117 | n=200: 0.7281 | avg 0.6763  (633s)
H_weights  n=5: 0.5879 | n=25: 0.6606 | n=50: 0.6942 | n=100: 0.7117 | n=200: 0.7287 | avg 0.6766  (430s)
E_boost    n=5: 0.5871 | n=25: 0.6628 | n=50: 0.6962 | n=100: 0.7121 | n=200: 0.7285 | avg 0.6773  (511s)
A_rf       n=5: 0.5888 | n=25: 0.6610 | n=50: 0.6937 | n=100: 0.7109 | n=200: 0.7256 | avg 0.6760  (430s)

sanity check vs notebook 05: largest difference 0.000158 (rounding to 4 decimals in 05's file)
done in 2923s


## 3. Verdict per idea (rule fixed in advance)

In [4]:
def compare(df_new, df_base, label):
    """Paired comparison per n: mean difference and wins over the 10 seeds."""
    m = df_new.merge(df_base, on=["n_per_class", "seed"], suffixes=("", "_base"))
    m["diff"] = m.f1 - m.f1_base
    g = m.groupby("n_per_class").agg(new=("f1", "mean"), base=("f1_base", "mean"), diff=("diff", "mean"),
                                     wins=("diff", lambda d: int((d > 0).sum())))
    passed = ((g.loc[25, "diff"] >= 0.005) or (g["diff"].mean() >= 0.005)) and (g["diff"].min() >= -0.005)
    return {"idea": label, **{f"diff_n{n}": g.loc[n, "diff"] for n in SHOTS}, "avg_diff": g["diff"].mean(),
            "wins_n25": g.loc[25, "wins"], "avg_new": g["new"].mean(), "rule_pass": bool(passed)}

def pick(config, prior=False, smooth=MAIN):
    s = runs[(runs.config == config) & (runs.prior == prior) & (runs.smooth == f"{smooth[0]}_{smooth[1]}")]
    return s[["n_per_class", "seed", "f1"]]

base = pick("baseline")
verdict = [compare(pick(c), base, c) for c in stage_cfgs if c != "baseline"]
verdict.append(compare(pick("baseline", prior=True), base, "C_prior"))
for a, rad in ex.SMOOTH_VARIANTS:
    if (a, rad) != MAIN:
        verdict.append(compare(pick("baseline", smooth=(a, rad)), base, f"B_smooth_{a}_{rad}"))
verdict = pd.DataFrame(verdict)

# extra Madrid gates for ideas that change Stage 1
verdict.loc[verdict.idea == "A_rf", "rule_pass"] &= A_madrid_ok
verdict.loc[verdict.idea == "H_weights", "rule_pass"] &= H_madrid_ok
# smoothing variants must also not hurt Madrid spatial CV
for i, r in verdict[verdict.idea.str.startswith("B_smooth")].iterrows():
    a, rad = r.idea.split("_")[2:]
    col = f"spatial_smooth_{a}_{rad}" if float(a) > 0 else "spatial_cv"
    verdict.loc[i, "rule_pass"] &= bool(b[col] >= b["spatial_smooth_0.5_1"] - 0.002)
# only the best smoothing variant can be adopted
B_cands = verdict[verdict.idea.str.startswith("B_smooth") & verdict.rule_pass]
B_choice = None
if len(B_cands):
    B_choice = B_cands.sort_values("avg_diff").iloc[-1].idea
    a, rad = B_choice.split("_")[2:]
    B_choice = (float(a), int(rad))

display(verdict.round(4))
verdict.round(4).to_csv(OUT / "verdict_per_idea.csv", index=False)
passed_stage = [c for c in stage_cfgs if c != "baseline" and verdict.set_index("idea").loc[c, "rule_pass"]]
C_pass = bool(verdict.set_index("idea").loc["C_prior", "rule_pass"])
print("PASSED (stage ideas):", passed_stage or "none", "| C prior:", C_pass, "| smoothing:", B_choice or "keep (0.5, 3x3)")

,idea,diff_n5,diff_n25,diff_n50,diff_n100,diff_n200,avg_diff,wins_n25,avg_new,rule_pass
0,D_align,0.0165,-0.0007,-0.0053,-0.0046,-0.0035,0.0005,5,0.6769,False
1,I_year,0.0001,-0.0001,-0.0005,-0.0001,-0.0002,-0.0002,5,0.6763,False
2,H_weights,0.0025,-0.0014,-0.0005,-0.0001,0.0004,0.0002,2,0.6766,False
3,E_boost,0.0017,0.0008,0.0015,0.0002,0.0002,0.0009,6,0.6773,False
4,A_rf,0.0035,-0.0010,-0.0010,-0.0009,-0.0026,-0.0004,5,0.6760,False
5,C_prior,-0.1794,-0.0299,-0.0133,-0.0078,-0.0063,-0.0474,0,0.6291,False
6,B_smooth_0.0_1,-0.0098,-0.0159,-0.0178,-0.0186,-0.0172,-0.0158,0,0.6606,False
7,B_smooth_0.5_2,0.0036,0.0066,0.0064,0.0060,0.0053,0.0056,10,0.6820,True
8,B_smooth_0.7_1,-0.0006,-0.0014,-0.0010,-0.0020,-0.0032,-0.0016,0,0.6748,False
9,B_smooth_0.7_2,0.0035,0.0073,0.0064,0.0064,0.0040,0.0055,10,0.6819,True


PASSED (stage ideas): none | C prior: False | smoothing: (0.5, 2)


## 4. Combine the winners, then confirm ONCE on the hidden neighbourhoods
If several ideas passed, they are combined and checked on the 80% split (the combination must not be worse than the best single idea by more than 0.002, otherwise the best single idea is used). Then the chosen setup and the baseline are both scored on the hidden 20% (never used for choosing). Baseline must reproduce notebook 05's hidden scores (sanity check).

In [5]:
t0 = time.time()
final_smooth = B_choice or MAIN
if not passed_stage and not C_pass and B_choice is None:
    print("Nothing passed: the frozen method stays. Nothing to confirm.")
    FINAL = None
else:
    cfg = {}
    for c in passed_stage:
        cfg.update(stage_cfgs[c])
    final_name = "+".join(passed_stage) or "baseline"
    if len(passed_stage) > 1:
        combo = screen("combined", cfg)
        runs = pd.concat([runs, combo], ignore_index=True)
        best_single = max(passed_stage, key=lambda c: verdict.set_index("idea").loc[c, "avg_new"])
        c_avg = compare(pick("combined", C_pass, final_smooth), base, "combined")["avg_new"]
        s_avg = compare(pick(best_single, C_pass, final_smooth), base, best_single)["avg_new"]
        print(f"combined avg {c_avg:.4f} vs best single ({best_single}) {s_avg:.4f}")
        if c_avg < s_avg - 0.002:
            cfg, final_name = stage_cfgs[best_single], best_single
    FINAL = {"stage": cfg, "name": final_name, "prior": C_pass, "smooth": final_smooth}
    print("candidate to confirm:", FINAL, f"({time.time()-t0:.0f}s)")

candidate to confirm: {'stage': {}, 'name': 'baseline', 'prior': False, 'smooth': (0.5, 2)} (0s)


In [6]:
def hidden_features(city):
    raw = dp.load_city(R / f"hidden_{city.lower()}.parquet")
    ev = {"Madrid": 1960, "Amsterdam": 1945}[city]
    f = fv3.build_features(dp.add_age_class(dp.clean_long_table(raw, combine="median"), ev))[0]
    return f[D["cols"]].to_numpy(float), f["age_class"].to_numpy(int), f[["px_key", "py_key"]].to_numpy(np.int64)

def hidden_eval(stage_cfg, prior, smooth, label):
    """Train exactly as in screening (80% split), score the hidden Amsterdam blocks (+ Madrid Stage 1).
    Smoothing uses the Amsterdam training pixels as context, like predict.py."""
    S = ex.build_stage1(D, stage_cfg)
    rows = []
    for n in SHOTS:
        for k in range(TRIALS):
            sup, qry, _, parts = ex.transfer_run(D, S, n, SEED + k, return_parts=True)
            Ph = ex.predict_new(S, parts, Xh)
            Pc = ex.predict_new(S, parts, D["Xa"])
            if prior:
                Ph = ex.prior_correct(Ph)[0]                            # class mix of the (unlabelled) test pixels
            P = Ph
            if smooth[0] > 0:
                P = tv3.smooth_proba(np.vstack([Ph, Pc]), np.vstack([ids_h, D["ids_a"]]), alpha=smooth[0], r=smooth[1])[: len(Ph)]
            rows.append({"setup": label, "n_per_class": n, "seed": SEED + k,
                         "f1_raw": f1_score(yh, CLASSES[Ph.argmax(1)], average="macro"),
                         "f1": f1_score(yh, CLASSES[P.argmax(1)], average="macro")})
    # Madrid hidden, Stage 1 only (context = Madrid training pixels)
    Pm = S["rf"].predict_proba(Xmh)
    Pmc = S["rf"].predict_proba(D["Xm"])
    Pms = tv3.smooth_proba(np.vstack([Pm, Pmc]), np.vstack([ids_mh, D["ids_m"]]), alpha=max(smooth[0], 1e-9), r=smooth[1])[: len(Pm)]
    mad_f1 = f1_score(ymh, CLASSES[Pms.argmax(1)], average="macro")
    return pd.DataFrame(rows), mad_f1

if FINAL is not None:
    t0 = time.time()
    Xh, yh, ids_h = hidden_features("Amsterdam")
    Xmh, ymh, ids_mh = hidden_features("Madrid")
    hb, mad_b = hidden_eval({}, False, MAIN, "baseline")
    # sanity check vs notebook 05 hidden scores (same models, same pixels)
    ref = pd.read_csv(R / "dress_rehearsal_all_runs.csv")[["n_per_class", "seed", "f1_hidden"]]
    gap = (hb.merge(ref, on=["n_per_class", "seed"]).eval("f1_raw - f1_hidden").abs().max())
    print(f"sanity check vs notebook 05 hidden: largest difference {gap:.6f}")
    assert gap < 1e-3, "baseline does not reproduce notebook 05 on the hidden blocks"
    hf, mad_f = hidden_eval(FINAL["stage"], FINAL["prior"], FINAL["smooth"], FINAL["name"])
    conf = compare(hf[["n_per_class", "seed", "f1"]], hb[["n_per_class", "seed", "f1"]], FINAL["name"])
    table = (pd.concat([hb, hf]).groupby(["setup", "n_per_class"]).f1.agg(["mean", "std"]).unstack(0).round(4))
    display(table)
    pd.concat([hb, hf]).round(4).to_csv(OUT / "hidden_confirmation_all_runs.csv", index=False)
    confirmed = conf["avg_diff"] >= 0 and conf["diff_n25"] >= -0.005 and conf["diff_n200"] >= -0.005
    print(f"Madrid hidden (Stage 1 + smoothing): baseline {mad_b:.4f} | new {mad_f:.4f}")
    print(f"Amsterdam hidden average: baseline {hb.groupby('n_per_class').f1.mean().mean():.4f} | "
          f"new {hf.groupby('n_per_class').f1.mean().mean():.4f} | n=25 diff {conf['diff_n25']:+.4f} "
          f"(wins {conf['wins_n25']}/{TRIALS})")
    print("FINAL VERDICT:", f"ADOPT {FINAL}" if confirmed else "NOT CONFIRMED on hidden neighbourhoods -> keep frozen method")
    json.dump({"final": {k: (str(v) if k == "stage" else v) for k, v in FINAL.items()}, "confirmed": bool(confirmed),
               "madrid_hidden_baseline": mad_b, "madrid_hidden_new": mad_f}, open(OUT / "final_verdict.json", "w"), indent=2)
    print(f"done in {time.time()-t0:.0f}s")

temporal haze filter removed 930 photos (0.31% of clean photos)
built 3,477 pixels x 132 features (0 NaN values filled with the city median)
temporal haze filter removed 5,485 photos (0.33% of clean photos)
built 14,532 pixels x 132 features (0 NaN values filled with the city median)
sanity check vs notebook 05 hidden: largest difference 0.000542


,mean,std
setup,baseline,baseline
n_per_class,,
5,0.5169,0.0426
25,0.5888,0.0309
50,0.6240,0.0182
100,0.6420,0.0107
200,0.6482,0.0071


Madrid hidden (Stage 1 + smoothing): baseline 0.6088 | new 0.6173
Amsterdam hidden average: baseline 0.6011 | new 0.6069 | n=25 diff +0.0038 (wins 7/10)
FINAL VERDICT: ADOPT {'stage': {}, 'name': 'baseline', 'prior': False, 'smooth': (0.5, 2)}
done in 956s
